# MiniMind-O 最小 profile（教学）

同一条 `Omni.generate`，用三件套看三层：

1. **torch.profiler / Kineto** — op 与 kernel 名字、stage 墙时
2. **nsys** — CUDA API + NVTX 阶段时间线
3. **ncu** — 单个 kernel 的 SOL / occupancy

阶段标签已经在 `nanovllm_omni/models/minimind_omni/_stage.py` 里打好
（`tokenize` / `generate` / `generate.step` / `decode` / `wav`）。
本 notebook **不再**包一层 `record_function`。

梯子：先 Kineto 定位烫的名字 → nsys 看是不是 launch-bound → 只对 **一个** kernel 开 ncu。
目标机：WSL RTX 3050 4GB。`max_tokens=16`、一条短 prompt。
不要 `import` bench CLI；教学目的是 10 行 profiler API。


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import torch
from nanovllm_omni import Omni, SamplingParams

# notebook 可从仓库根或 notebooks/ 启动
REPO = Path.cwd() if (Path.cwd() / "nanovllm_omni").is_dir() else Path.cwd().parent
OUT = REPO / "notebooks" / "out"
OUT.mkdir(parents=True, exist_ok=True)

MODEL = "jingyaogong/minimind-3o"  # 本地权重改成 pretrained/minimind-3o
PROMPT = ["你好"]
SAMPLING = SamplingParams(max_tokens=16, temperature=0.7, top_p=0.9, seed=42)

omni = Omni(MODEL)
# warmup：JIT / autotune / cuDNN benchmark 留在 profiler 外面
_ = omni.generate(PROMPT, SAMPLING)
if torch.cuda.is_available():
    torch.cuda.synchronize()
print("ready", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("out ->", OUT)


## 1. torch.profiler（进程内）

Kineto 回答「名字」。导出 Chrome trace 后用 `chrome://tracing` 或 https://ui.perfetto.dev 打开，泳道里找 `generate` / `generate.step`。

warmup 已在上一格跑完。看表时三件事：

- `cudaLaunchKernel` 次数巨大 → 先怀疑 launch 税，不要急着写 kernel
- `direct_copy_kernel_cuda` 的 **count 是代码形状**（16 step 下曾稳定 1744），不是噪声
- chrome 里应能看到 `stage()` 打的 `user_annotation`；没有则说明没走 MiniMind-O 这条路径


In [ ]:
from torch.profiler import ProfilerActivity, profile

trace_path = OUT / "torch.json"

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=False,
) as prof:
    out = omni.generate(PROMPT, SAMPLING)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

prof.export_chrome_trace(str(trace_path))
wav = out[0].multimodal_output["audio"].wav_bytes()
print("trace ->", trace_path, "wav bytes", len(wav))

rows = [
    (ev.key, ev.self_device_time_total / 1e3, ev.count)
    for ev in prof.key_averages()
    if ev.device_type == torch.autograd.DeviceType.CUDA
]
rows.sort(key=lambda r: -r[1])
print(f"{'kernel':<60} {'ms':>8} {'n':>6}")
for name, ms, n in rows[:15]:
    print(f"{name:<60} {ms:8.2f} {n:6d}")


## 2. nsys（必须包子进程）

Jupyter 已经在跑，`nsys profile` 包不到当前 kernel。写出与上一格同一份 `generate` 的 inner，再 `nsys profile ... python inner.py`。

关键 argv（抄 `nanovllm_omni/optim/bench/profile.py`）：

- `-t cuda,nvtx` — 没有 `nvtx` 就看不到 `stage()`
- `--force-overwrite=true`
- `-s none` — 关掉 CPU sampling，3050 上更干净

`stage()` 之后 NVTX 应出现 `:tokenize` / `:generate` / `:decode` / `:wav` / `:generate.step`。
WSL 上 CUPTI 可能 `cuda_gpu_kern_sum SKIPPED`：系统时间线还在，GPU kernel duration 可能没有。


In [ ]:
inner = OUT / "inner.py"
inner.write_text(
    (
        "from nanovllm_omni import Omni, SamplingParams\n"
        f"omni = Omni({MODEL!r})\n"
        f"omni.generate({PROMPT!r}, SamplingParams("
        "max_tokens=16, temperature=0.7, top_p=0.9, seed=42))\n"
    ),
    encoding="utf-8",
)
print("inner ->", inner)
print(inner.read_text())


In [ ]:
nsys = shutil.which("nsys")
if not nsys:
    print("nsys 不在 PATH：sudo apt install -y nsight-systems-cli")
else:
    nsys_out = OUT / "nsys"
    cmd = [
        nsys, "profile",
        "-o", str(nsys_out),
        "-t", "cuda,nvtx",
        "--force-overwrite=true",
        "-s", "none",
        sys.executable, str(inner),
    ]
    print(" ".join(cmd))
    subprocess.check_call(cmd)
    print("rep ->", str(nsys_out) + ".nsys-rep")


In [ ]:
rep = OUT / "nsys.nsys-rep"
nsys = shutil.which("nsys")
if not nsys or not rep.exists():
    print("skip nsys stats（没有 nsys 或还没生成 .nsys-rep）")
else:
    cmd = [nsys, "stats", "--report", "nvtx_sum", "--format", "csv", str(rep)]
    print(" ".join(cmd))
    subprocess.check_call(cmd)


## 3. ncu（只打一个 kernel）

全量 ncu 会把 3050 上一次 generate 拉成几十分钟。这里只抓 `gemv2T` 的 3 个稳态 instance。

两个坑（仓库里踩过）：

1. **不要加 `--profile-from-start off`**。这版 ncu 上它不是「晚点再开」，而是一个 kernel 都不采。
2. **WSL 3050 经常 `ERR_NVGPUCTRPERM`**。失败本身就是结论：ncu 依赖驱动权限，torch.profiler 不依赖。不要改 notebook 逻辑去「修好」它。

读报告：`ncu --import notebooks/out/ncu-gemv.ncu-rep --csv --page raw`
看 Compute SOL / Memory SOL / grid。SOL 个位数 + grid 填不满 SM → launch-bound。


In [ ]:
ncu = shutil.which("ncu")
if not ncu:
    print("ncu 不在 PATH：装 Nsight Compute CLI")
elif not inner.exists():
    print("先跑 nsys 那一格写 inner.py")
else:
    ncu_out = OUT / "ncu-gemv"
    cmd = [
        ncu,
        "--target-processes", "all",
        "--kernel-name", r"regex:.*gemv2T.*",
        "--launch-skip", "50",
        "--launch-count", "3",
        "--section", "SpeedOfLight,Occupancy,LaunchStats",
        "--export", str(ncu_out),
        "--force-overwrite",
        sys.executable, str(inner),
    ]
    print(" ".join(cmd))
    rc = subprocess.call(cmd)
    if rc != 0:
        print(
            f"ncu rc={rc}。WSL 3050 常见 ERR_NVGPUCTRPERM；"
            "换有计数器权限的机器或关 NVreg_RestrictProfilingToAdminUsers。"
        )
    else:
        print("rep ->", str(ncu_out) + ".ncu-rep")
